In [ ]:
%load_ext autoreload
%autoreload 2
%cd /opt/tiger/samantha

from recipes.research import seed_everything

seed_everything(seed=42, workers=True)

In [ ]:

from recipes.research.dataset.collection import (
    EveryNoiseParquetDataset,
    PlaylistV5ParquetDataset,
    ShutterStockFeatureParquetDataset,
    ShutterStockParquetDataset,
    MultiLingualBigASR48LangParquetDataset,
    Billboardv2WebDataset,
    MusDBTrainAudioFolderDataset
)

In [ ]:
batch_size = 64
sample_rate = 44100
channels = 2
segment_duration = 30
num_workers = 8

playlist = PlaylistV5ParquetDataset(
    sample_rate=sample_rate,
    channels=channels,
    segment_duration=segment_duration,
    resampled=True,
    shardshuffle=True,
)

shutterstock = ShutterStockParquetDataset(
    sample_rate=sample_rate,
    channels=channels,
    segment_duration=segment_duration,
    resampled=True,
    shardshuffle=True,
)

everynoise = EveryNoiseParquetDataset(
    sample_rate=sample_rate,
    channels=channels,
    segment_duration=segment_duration,
    resampled=True,
    shardshuffle=True,
)

billboard_v2 = Billboardv2WebDataset(
    sample_rate=sample_rate,
    channels=channels,
    segment_duration=segment_duration,
    resampled=True,
    shardshuffle=True,
)

musdb_train = MusDBTrainAudioFolderDataset(
    sample_rate=sample_rate,
    channels=channels,
    segment_duration=segment_duration
)

In [ ]:
from samantha.data.audio.dataset import AudioFolderDataModule

datamodule = AudioFolderDataModule(
    [playlist, shutterstock, everynoise, billboard_v2, musdb_train],
    [],
    [],
    weights=None,
    batch_size=batch_size,
    shuffle=None,
    num_workers=num_workers,
    prefetch_factor=2,
    batch_drop_duplicates=True,
)
dataloader = datamodule.train_dataloader()
dataloader = iter(dataloader)

In [ ]:
batch = next(dataloader)


In [ ]:
from IPython.display import Audio, display
from samantha.data.av_audio import audio_write

for idx in range(batch_size):
    print(batch.key[idx], batch.shard[idx])
    print(batch.segment_info[idx].seek_time, batch.segment_info[idx].meta.duration)
    
    mp3_bytes = audio_write(batch.audio[idx], sample_rate, format="mp3")
    display(Audio(mp3_bytes))

In [ ]:
import torch
frame_rate = 25
lengths = torch.tensor(
    [
        (s.n_frames / s.sample_rate) * frame_rate
        for s in batch.segment_info
    ],
    device=batch.audio.device,
).floor()
lengths